# 🔮 Crystal Mancer — Scaled-Up Data Collection + Mining

This notebook runs the **full data pipeline** on Google Colab:

| Step | What | Scale |
|------|------|-------|
| 1 | Materials Project ALL oxides | ~150K+ structures |
| 2 | GNoME stable crystals + CIFs | 79K records, 50K CIFs |
| 3 | Literature mining (31 queries × 200 papers) | ~3K+ papers |
| 4 | LLM extraction via OpenRouter | regex + LLM |
| 5 | Literature enrichment + canonical dedup | Unified dataset |

**Data location:** `My Drive / CrystalMancerData /`  
**Resume-safe:** If Colab disconnects, re-run — it picks up where it left off.

---

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

GDRIVE_DATA_DIR = '/content/drive/MyDrive/CrystalMancerData'
os.makedirs(GDRIVE_DATA_DIR, exist_ok=True)
print(f'✅ Data will be saved to: {GDRIVE_DATA_DIR}')

## 2. Clone Repo + Install Dependencies

In [ ]:
import os

# Clone or pull latest
if os.path.exists('crystalMancer'):
    !cd crystalMancer && git pull
else:
    !git clone https://github.com/rosn2112/crystalMancer.git

%cd crystalMancer

# Install all dependencies
%pip install -q mp-api pymatgen requests pandas numpy tqdm

## 3. Set API Keys

**Required:** Materials Project API key  
**Optional:** OpenRouter key for LLM-powered extraction (free tier)

In [ ]:
import os

# ── Required: Materials Project API key ──
os.environ['MP_API_KEY'] = 'V9oxuqCRq4MvdyDAzrE2FYLlvmw29wjz'

# ── Optional: OpenRouter key for LLM extraction ──
# Get a free key at: https://openrouter.ai/keys
# Uncomment and paste your key below:
# os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-YOUR_KEY_HERE'

print('✅ API keys set.')
if os.environ.get('OPENROUTER_API_KEY'):
    print('   🤖 LLM extraction ENABLED')
else:
    print('   📝 Regex extraction only (set OPENROUTER_API_KEY for LLM)')

## 4. Run the Full Download + Mining Pipeline

This runs everything sequentially:

1. **Materials Project** — Downloads ALL oxide structures (~150K+)
2. **GNoME** — Downloads 79K records + CIFs for top 50K
3. **Literature Mining** — 31 queries × 200+ papers from 3 APIs
4. **Unified Dataset** — Merges, deduplicates, enriches

⏱️ **Expected time:** 2-4 hours (mostly MP download + literature APIs)

**Resume-safe:** If Colab disconnects, just re-run. Downloads skip existing data.

In [ ]:
!python scripts/download_all.py

## 5. Run Data Mining Pipeline (enrichment + dedup)

If you've already downloaded the structures and just need to:
- Mine more papers
- Enrich structures with literature data
- Rebuild the unified dataset

Run the orchestrator separately:

In [ ]:
# Option A: Full mining + enrichment
!python scripts/run_data_mining.py --full

# Option B: Only enrich existing data (skip mining)
# !python scripts/run_data_mining.py --enrich-only

## 6. Verify Data on Drive

In [ ]:
import os
import json

data_dir = '/content/drive/MyDrive/CrystalMancerData'

print('📊 CrystalMancerData Contents:')
print('=' * 60)
total_size = 0
for root, dirs, files in os.walk(data_dir):
    level = root.replace(data_dir, '').count(os.sep)
    indent = '  ' * level
    subdir = os.path.basename(root)
    n_files = len(files)
    dir_size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
    total_size += dir_size
    mb = dir_size / 1e6
    print(f'{indent}📂 {subdir}: {n_files} files ({mb:.1f} MB)')
    if level >= 2:
        continue

print(f'\n📦 Total: {total_size/1e9:.2f} GB')

# Show stats from latest run
stats_file = os.path.join(data_dir, 'mining_stats.json')
if os.path.exists(stats_file):
    with open(stats_file) as f:
        stats = json.load(f)
    print(f'\n📈 Latest Mining Stats:')
    if 'literature' in stats:
        lit = stats['literature']
        print(f"  Papers: {lit.get('total_papers', 0)}")
        print(f"  With performance data: {lit.get('with_performance', 0)}")
        if lit.get('reaction_types'):
            print(f"  Reaction types: {lit['reaction_types']}")
    if 'dataset' in stats:
        ds = stats['dataset']
        print(f"  Dataset records: {ds.get('total', 0)}")
        print(f"  By source: {ds.get('by_source', {})}")
        print(f"  With DFT energy: {ds.get('has_energy', 0)}")
        print(f"  With catalysis data: {ds.get('has_catalysis', 0)}")

## 7. Quick Dataset Preview

In [ ]:
import json
import pandas as pd

# Load unified dataset
data_dir = '/content/drive/MyDrive/CrystalMancerData'
dataset_file = os.path.join(data_dir, 'unified_dataset.jsonl')

records = []
with open(dataset_file) as f:
    for line in f:
        records.append(json.loads(line.strip()))

df = pd.DataFrame(records)
print(f'Total records: {len(df)}')
print(f'\nSource distribution:')
print(df['source'].value_counts())
print(f'\nSample records:')
display(df[['material_id', 'canonical_formula', 'source',
            'formation_energy_per_atom', 'band_gap']].head(10))

# Show enriched records
enriched = df[df['catalysis_data'].notna()]
print(f'\nRecords with catalysis data: {len(enriched)}')
if len(enriched) > 0:
    display(enriched[['material_id', 'canonical_formula', 'source']].head(10))